# Writer-Adaptive Similarity Metric Estimation

The contribution of this study, implemented and evaluated. Every existing offline verification
system compares signatures with one similarity function applied identically to every writer.
This notebook estimates, for each writer separately, how much each dissimilarity component
should count for that writer, using that writer's own enrolled genuine signatures and no
forgery at any stage.

**The idea.** A writer who signs with a consistent slant but variable size should be judged
strictly on slant and loosely on size; a writer with the opposite habit should be judged the
other way round. The spread of a writer's own genuine-to-genuine dissimilarities along each
component measures how much variation that component tolerates for that writer. A component
with narrow spread is one the writer is consistent on and is weighted heavily; a component
with wide spread is discounted.

**Why it is possible at twelve signatures.** Twelve signatures give twelve vectors in a
feature space of thousands of dimensions, which cannot support estimation of anything. The
same twelve give 66 pairs described in 17 dimensions, which can.

**Conditions compared.** Four, all through an identical pipeline and protocol.

| | Condition | Tests |
| --- | --- | --- |
| A | Per-writer estimated metric | the proposal |
| B | Single global metric, pooled over development writers | H1 |
| C | Unweighted metric | the untuned reference |
| D | Nine trivial image statistics | H2 |

**Hypotheses.** H1: a per-writer metric reduces verification error relative to one global
metric. H2: the proposed representation outperforms non-signature statistics through an
identical pipeline. Both are tested by paired Wilcoxon signed-rank across writers with Holm
correction and bootstrap confidence intervals.

**Yardstick from v8.** At the model's input, trivial statistics already reach mean per-writer
AUC 0.77 on CEDAR, 0.76 on Hindi and 0.91 on Bengali. Condition A must be read against
condition D on the same writers, never on its own.

In [ ]:
pip install numpy pandas pillow scikit-image scikit-learn scipy matplotlib notebook

In [ ]:
# ---- Configuration -------------------------------------------------------
import os, re, json, math, warnings, itertools
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

LOCAL_BASE = r"C:\Users\___\Downloads\bhsig\bhsig260-dataset"
COLAB_BASE = "/content/drive/MyDrive/bhsig260-dataset"
CORPORA = {"CEDAR":"cedar", "Hindi":"bhsig260-hindi", "Bengali":"bhsig260-bengali"}

SPLIT_SEED    = 20261014     # unchanged from v4a
DEV_FRACTION  = 0.45
RANDOM_STATE  = 0
N_ENROL       = 12           # enrolled genuine signatures per writer
N_TEST_GEN    = 12           # held-out genuine per writer
N_TEST_FOR    = 12           # forged per writer
TARGET_SIZE   = (100, 100)
MAX_ASPECT    = 2.5
DENSITY_KEEP  = 0.02
EPS           = 1e-9
N_BOOT        = 2000
FEATURE_CACHE = "v9_features.npz"
RUN_CONFIRMATION = False     # set True once, at the very end, to touch the reserved writers

def resolve_base():
    for cand in (LOCAL_BASE, os.path.join(os.getcwd(), "bhsig260-dataset")):
        if os.path.isdir(cand): return cand, "local"
    if os.path.isdir(COLAB_BASE): return COLAB_BASE, "colab (already mounted)"
    try:
        from google.colab import drive; drive.mount("/content/drive")
        if os.path.isdir(COLAB_BASE): return COLAB_BASE, "colab"
    except ImportError: pass
    raise FileNotFoundError(f"Dataset not found. tried {LOCAL_BASE} and {COLAB_BASE}")

BASE, WHERE = resolve_base()
print("numpy", np.__version__, "| pandas", pd.__version__, "| running on:", WHERE)
try: display
except NameError: from IPython.display import display

## 1. Corpora, manifest and the writer split

Identical to v8: the same discovery, the same filename conventions, the same seed and the same
development fraction, so that the writers used here are exactly the writers audited there.

In [ ]:
IMG_EXT = {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
RE_CEDAR = re.compile(r"^(original|forgeries)[_-](\d+)[_-](\d+)", re.I)
RE_BHSIG = re.compile(r"^([BH])-S-(\d+)-([GF])-(\d+)", re.I)

def find_root(base, name):
    name = name.lower()
    def scan(folder):
        try: entries=[d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder,d))]
        except OSError: return None
        for d in entries:
            if d.lower()==name: return os.path.join(folder,d)
        for d in entries:
            if d.lower().startswith(name): return os.path.join(folder,d)
        return None
    hit=scan(base)
    if hit: return hit
    for d in sorted(os.listdir(base)):
        p=os.path.join(base,d)
        if os.path.isdir(p):
            hit=scan(p)
            if hit: return hit
    return None

def parse(path, corpus):
    stem=os.path.splitext(os.path.basename(path))[0]
    m=RE_CEDAR.match(stem)
    if m: return dict(writer=f"{corpus}-{int(m.group(2)):03d}",
                      label="genuine" if m.group(1).lower()=="original" else "forged",
                      index=int(m.group(3)))
    m=RE_BHSIG.match(stem)
    if m: return dict(writer=f"{corpus}-{int(m.group(2)):03d}",
                      label="genuine" if m.group(3).upper()=="G" else "forged",
                      index=int(m.group(4)))
    return None

rows=[]
for key, folder in CORPORA.items():
    root=find_root(BASE, folder)
    if not root: print("MISSING corpus root:", key); continue
    for dp,_,fs in os.walk(root):
        for f in fs:
            if os.path.splitext(f)[1].lower() in IMG_EXT:
                p=os.path.join(dp,f); rec=parse(p,key)
                if rec: rec.update(corpus=key, path=p,
                                   rel=os.path.relpath(p,BASE).replace("\\","/")); rows.append(rec)
manifest=pd.DataFrame(rows)

pw=manifest.groupby(["corpus","writer","label"]).size().unstack(fill_value=0)
bad=pw[(pw.get("genuine",0)<N_ENROL+1)|(pw.get("forged",0)<1)]
if len(bad):
    print("writers excluded for insufficient signatures:", len(bad))
    manifest=manifest[~manifest.writer.isin(bad.index.get_level_values("writer"))]

rng=np.random.RandomState(SPLIT_SEED); dev_w=[]
for corpus,grp in manifest.groupby("corpus"):
    ws=np.array(sorted(grp.writer.unique())); perm=rng.permutation(len(ws))
    dev_w.extend(ws[perm[:max(2,int(round(DEV_FRACTION*len(ws))))]].tolist())
dev_w=set(dev_w)
manifest["split"]=np.where(manifest.writer.isin(dev_w),"dev","reserved")
print(manifest.groupby(["corpus","split"])["writer"].nunique().unstack(fill_value=0))
print("total images:", len(manifest))

## 2. Preprocessing and features

Preprocessing is the S0 to S4 chain audited in v8, ending at a 100 by 100 image. Three feature
representations are extracted: a Histogram of Oriented Gradients descriptor, raw pixels, and
the nine-statistic control that carries no information about signature shape.

The HOG implementation must be the one used from v3 onward for results to be comparable; the
cell below uses scikit-image and reports the parameters it ran with.

In [ ]:
from PIL import Image

def otsu_threshold(a):
    hist=np.bincount(a.ravel(),minlength=256).astype(np.float64); tot=hist.sum()
    if tot==0: return 127
    om=np.cumsum(hist)/tot; mu=np.cumsum(hist*np.arange(256))/tot; mut=mu[-1]
    den=om*(1-om); den[den==0]=1e-12
    return int(np.argmax((mut*om-mu)**2/den))

def load_gray(path):
    with Image.open(path) as im:
        if im.mode not in ("L","I;16","I"): im=im.convert("L")
        a=np.asarray(im)
    if a.dtype!=np.uint8:
        a=(255.0*(a.astype(np.float64)-a.min())/max(1e-9,(a.max()-a.min()))).astype(np.uint8)
    return a

def preprocess(path):
    a=load_gray(path); a=(255-a).astype(np.uint8)
    t=otsu_threshold(a); a=np.where(a>t,a,0).astype(np.uint8)
    r=a.sum(1).astype(float); c=a.sum(0).astype(float)
    rm=r.max() or 1.0; cm=c.max() or 1.0
    ri=np.where(r>=DENSITY_KEEP*rm)[0]; ci=np.where(c>=DENSITY_KEEP*cm)[0]
    b=a[ri.min():ri.max()+1, ci.min():ci.max()+1] if len(ri) and len(ci) else a
    h,w=b.shape[:2]
    if h and w:
        if w/h>MAX_ASPECT:
            nh=int(np.ceil(w/MAX_ASPECT)); p=nh-h; b=np.pad(b,((p//2,p-p//2),(0,0)))
        elif h/w>MAX_ASPECT:
            nw=int(np.ceil(h/MAX_ASPECT)); p=nw-w; b=np.pad(b,((0,0),(p//2,p-p//2)))
    return np.asarray(Image.fromarray(b.astype(np.uint8)).resize(TARGET_SIZE, Image.BILINEAR))

try:
    from skimage.feature import hog as sk_hog
    HOG_KW=dict(orientations=9, pixels_per_cell=(10,10), cells_per_block=(2,2),
                block_norm="L2-Hys", feature_vector=True)
    def feat_hog(img): return sk_hog(img, **HOG_KW).astype(np.float32)
    print("HOG via scikit-image:", HOG_KW)
except ImportError:
    raise ImportError("scikit-image is required for HOG. Install it, then re-run: pip install scikit-image")

def feat_raw(img): return (img.astype(np.float32)/255.0).ravel()

def feat_stats(img):
    a=img.astype(np.float64); t=otsu_threshold(img); h,w=img.shape[:2]
    return np.array([a.mean(), a.std(), np.median(a), np.percentile(a,5), np.percentile(a,95),
                     float((img<=t).mean()), float(h), float(w), float(w)/float(h)], dtype=np.float32)

FEATS={"HOG":feat_hog, "RAW":feat_raw, "CTRL":feat_stats}
print("representations:", list(FEATS))

In [ ]:
# Select the signatures each writer contributes, then extract features once and cache.
def writer_sets(df):
    g=df[df.label=="genuine"].sort_values("index")
    f=df[df.label=="forged"].sort_values("index")
    return (g.path.tolist()[:N_ENROL],
            g.path.tolist()[N_ENROL:N_ENROL+N_TEST_GEN],
            f.path.tolist()[:N_TEST_FOR])

SETS={}
for (corpus,writer),grp in manifest.groupby(["corpus","writer"]):
    en,tg,tf = writer_sets(grp)
    if len(en)==N_ENROL and len(tg)>=1 and len(tf)>=1:
        SETS[(corpus,writer,grp.split.iloc[0])]=(en,tg,tf)
print("usable writers:", len(SETS), "| dev:", sum(1 for k in SETS if k[2]=="dev"))

all_paths=sorted({p for v in SETS.values() for lst in v for p in lst})
print("images to featurise:", len(all_paths))

if os.path.exists(FEATURE_CACHE):
    z=np.load(FEATURE_CACHE, allow_pickle=True)
    FEATURES={k:z[k].item() for k in FEATS}
    have=set(next(iter(FEATURES.values())).keys())
    print("cache loaded:", len(have), "images")
else:
    FEATURES={k:{} for k in FEATS}; have=set()

todo=[p for p in all_paths if p not in have]
print("to extract:", len(todo))
for i,p in enumerate(todo,1):
    img=preprocess(p)
    for k,fn in FEATS.items(): FEATURES[k][p]=fn(img)
    if i%500==0: print(f"  {i}/{len(todo)}")
if todo:
    np.savez_compressed(FEATURE_CACHE, **{k:np.array(v,dtype=object) for k,v in FEATURES.items()})
    print("cache written:", FEATURE_CACHE)
for k in FEATS:
    d=next(iter(FEATURES[k].values())); print(f"  {k:5s} dim {np.asarray(d).size}")

## 3. The pairwise dissimilarity representation

Each pair of signatures is described by seventeen symmetric dissimilarity components. Symmetry
matters: the descriptor of the pair (a, b) must equal that of (b, a), since no ordering of two
enrolled signatures is meaningful. Each component has a stated geometric meaning, which is what
makes the estimated weighting interpretable and what allows attribution at the component level.

Twelve enrolled signatures give C(12,2) = 66 such vectors in 17 dimensions, which is 3.9 samples
per dimension. The same twelve signatures in HOG space give 0.003. The representation is not the
contribution; it is what makes the contribution estimable.

In [ ]:
PAIR_COMPONENTS = [
    "L1","L2","Linf","cosine","correlation","canberra","braycurtis",
    "absdiff_std","absdiff_median","absdiff_p90",
    "d_mean","d_std","d_max","d_energy","chi2","hist_inter","d_sum"]
D_PAIR = len(PAIR_COMPONENTS)

def pair_descriptor(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    d = np.abs(a - b)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    cos = 1.0 - float(a @ b) / (na * nb + EPS)
    ac, bc = a - a.mean(), b - b.mean()
    cor = 1.0 - float(ac @ bc) / (np.linalg.norm(ac) * np.linalg.norm(bc) + EPS)
    can = float(np.sum(d / (np.abs(a) + np.abs(b) + EPS)))
    bray = float(d.sum() / (np.abs(a + b).sum() + EPS))
    chi2 = float(np.sum(d**2 / (np.abs(a) + np.abs(b) + EPS)))
    inter = float(np.minimum(np.abs(a), np.abs(b)).sum())
    hist = 1.0 - inter / (max(np.abs(a).sum(), np.abs(b).sum()) + EPS)
    return np.array([
        float(d.mean()), float(np.sqrt((d**2).sum())), float(d.max()),
        cos, cor, can, bray,
        float(d.std()), float(np.median(d)), float(np.percentile(d, 90)),
        abs(float(a.mean() - b.mean())), abs(float(a.std() - b.std())),
        abs(float(a.max() - b.max())), abs(float((a**2).sum() - (b**2).sum())),
        chi2, hist, abs(float(a.sum() - b.sum()))], dtype=np.float64)

def enrolled_pairs(vecs):
    return np.array([pair_descriptor(vecs[i], vecs[j])
                     for i, j in itertools.combinations(range(len(vecs)), 2)])

def query_pairs(q, vecs):
    return np.array([pair_descriptor(q, v) for v in vecs])

# symmetry check
_a = np.random.RandomState(0).rand(50); _b = np.random.RandomState(1).rand(50)
assert np.allclose(pair_descriptor(_a,_b), pair_descriptor(_b,_a)), "descriptor is not symmetric"
print(f"pair descriptor: {D_PAIR} components, symmetry verified")
print(f"  {N_ENROL} enrolled signatures -> {N_ENROL*(N_ENROL-1)//2} pairs"
      f" = {N_ENROL*(N_ENROL-1)//2/D_PAIR:.2f} samples per dimension")

## 4. Metric estimation

A writer's metric is a per-component scaling applied before the decision boundary is fitted,
expressed as a diagonal transformation so that each component keeps its meaning.

**Components are standardised against the population first.** The seventeen components are
sums, norms and percentiles over feature vectors of very different magnitude, so their raw
spreads differ by orders of magnitude. Weighting by the inverse of a raw spread would therefore
be dominated by whichever component happens to be smallest in absolute units, regardless of
whether it carries information. Every condition first divides each component by its spread over
the pooled development writers. A writer's own spread is then measured on those standardised
coordinates, so a weight above one means that writer is *more consistent than the population* on
that component, and a weight below one means less. This makes the weights comparable across
components and interpretable on sight.

The three conditions then differ in exactly one respect, the source of the per-component scale:

| Condition | Centre | Per-component scale |
| --- | --- | --- |
| A | the writer's own enrolled pairs | the writer's own, relative to the population |
| B | the writer's own enrolled pairs | the population's |
| C | the writer's own enrolled pairs | none; one shared scalar |

All three centre on the writer, because a one-class model is always about one writer; only the
scaling varies, so the comparison isolates the metric and nothing else.

**Two parts of a writer's metric.** The estimate splits into a profile, the normalised weighting
across components, and a scale, the overall tightness. Normalising the weights to mean one
removes the scale deliberately, since overall tightness is absorbed by the kernel bandwidth. Both
are reported, because the per-writer claim rests on the profile differing between writers and
that has to be measured rather than assumed.

In [ ]:
def fit_metric_global(P_all):
    """Population centre and per-component spread, pooled over development writers."""
    return P_all.mean(0), P_all.std(0) + EPS

def transform_for(P, condition, global_spread):
    """Return (transform, weights, scale) for one writer under one condition.

    Every condition centres on the writer and standardises components against the
    population first, so no component can dominate through its raw units alone.
    """
    mu_w = P.mean(0)
    if condition == "C":
        s = float(global_spread.mean())
        scale_vec = np.full(P.shape[1], s)
    elif condition == "B":
        scale_vec = global_spread.copy()
    else:                                  # A, and D on control features
        Z = (P - mu_w) / global_spread     # population-standardised coordinates
        sw = Z.std(0) + EPS                # this writer's spread, relative to the population
        scale_vec = global_spread * sw
    def tf(X): return (X - mu_w) / scale_vec
    if condition in ("A", "D"):
        w = 1.0 / ((P - mu_w).std(0) / global_spread + EPS)
    else:
        w = 1.0 / scale_vec
    return tf, w / (w.mean() + EPS), float(np.exp(np.mean(np.log(scale_vec + EPS))))

def metric_weights(w): return w
def metric_scale(s):   return s

print("metric estimation: components standardised against the population, then weighted per writer")
print("weight > 1 means the writer is more consistent than the population on that component")

## 5. Verification protocol

For a writer, the enrolled pairs are transformed by that writer's metric and a one-class Support
Vector Machine is fitted in the resulting space. A questioned signature is paired against each of
the twelve enrolled signatures, giving twelve decisions; the verdict is the majority, and the
continuous score is the mean decision value, which is what the area under the ROC curve is
computed from.

The kernel bandwidth is set by the scale of the data in each condition rather than fixed, because
the three conditions produce spaces of different scale and a fixed bandwidth would advantage
whichever one happened to match it. Non-degeneracy is verified rather than assumed: a bandwidth
far below the inverse of the typical squared distance makes every kernel evaluation approach
unity, at which point the kernel no longer measures distance and the fitted model is vacuous.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score

NU = 0.10

def kernel_health(X, gamma):
    n = min(len(X), 60)
    idx = np.random.RandomState(RANDOM_STATE).choice(len(X), n, replace=False)
    Z = X[idx]
    d2 = ((Z[:, None, :] - Z[None, :, :]) ** 2).sum(-1)
    iu = np.triu_indices(n, 1)
    return float(np.median(gamma * d2[iu]))

def evaluate_writer(key, rep, condition, global_centre, global_spread):
    corpus, writer, split = key
    enrol, tgen, tfor = SETS[key]
    F = FEATURES[rep]
    ev = [F[p] for p in enrol]
    P = enrolled_pairs(ev)
    tf, weights, scale = transform_for(P, condition, global_spread)

    Xtr = tf(P)
    gamma = 1.0 / (Xtr.shape[1] * (Xtr.var() + EPS))
    health = kernel_health(Xtr, gamma)
    m = OneClassSVM(kernel="rbf", nu=NU, gamma=gamma, tol=1e-10).fit(Xtr)

    scores, accepts = [], []
    for p in (tgen + tfor):
        s = m.decision_function(tf(query_pairs(F[p], ev)))
        scores.append(float(np.mean(s))); accepts.append(float(np.mean(s > 0)))
    scores = np.array(scores); verdict = np.array(accepts) > 0.5
    ngen = len(tgen)
    frr = float(1.0 - verdict[:ngen].mean())
    far = float(verdict[ngen:].mean())
    y = np.r_[np.zeros(ngen), np.ones(len(tfor))]
    try: auc = float(roc_auc_score(y, -scores))
    except Exception: auc = np.nan
    return dict(corpus=corpus, writer=writer, split=split, rep=rep, condition=condition,
                FRR=frr, FAR=far, AUC=auc, gamma=gamma, kernel_health=health,
                weights=weights, scale=scale)

print("protocol defined | nu =", NU, "| gamma adapted per condition")
print("condition D receives the same per-writer treatment as A, on control features")

## 6. Running the four conditions

The global metric is fitted on development writers' enrolled pairs only, so no reserved writer
influences any condition. Condition D repeats the whole procedure on the nine-statistic control,
which carries no information about signature shape; it is the yardstick that v8 showed sits at
mean per-writer AUC 0.75 to 0.91 at this exact input.

In [ ]:
DEV_KEYS = [k for k in SETS if k[2] == "dev"]
RES_KEYS = [k for k in SETS if k[2] == "reserved"]
print("development writers:", len(DEV_KEYS), "| reserved:", len(RES_KEYS))

PRIMARY_REP = "HOG"
CONDITIONS = [("A", PRIMARY_REP), ("B", PRIMARY_REP), ("C", PRIMARY_REP), ("D", "CTRL")]
# D is evaluated with the same per-writer metric as A, differing only in the features it reads.

def pool_global(keys, rep):
    P = [enrolled_pairs([FEATURES[rep][p] for p in SETS[k][0]]) for k in keys]
    return np.vstack(P)

GLOBALS = {}
for rep in {r for _, r in CONDITIONS}:
    Pall = pool_global(DEV_KEYS, rep)
    GLOBALS[rep] = fit_metric_global(Pall)
    print(f"global metric for {rep}: pooled {Pall.shape[0]} pairs")

rows = []
for cond, rep in CONDITIONS:
    gc, gs = GLOBALS[rep]
    for i, k in enumerate(DEV_KEYS, 1):
        rows.append(evaluate_writer(k, rep, cond, gc, gs))
        if i % 25 == 0: print(f"  {cond}/{rep}: {i}/{len(DEV_KEYS)} writers")
R = pd.DataFrame(rows)
W = R.pop("weights")
R["cond_label"] = R.condition.map({"A":"A per-writer metric","B":"B global metric",
                                   "C":"C unweighted","D":"D trivial control"})
print("rows:", len(R))

In [ ]:
# Kernel non-degeneracy: any condition whose kernel collapses is reported as failing, not as a result.
kh = R.groupby("cond_label")["kernel_health"].agg(["min","median","max"]).round(6)
display(kh)
DEGENERATE = R.kernel_health < 1e-3
print("writer-conditions with a degenerate kernel:", int(DEGENERATE.sum()), "of", len(R))
if DEGENERATE.any():
    display(R[DEGENERATE].groupby("cond_label").size().to_frame("degenerate"))

In [ ]:
# Headline table: every accuracy figure sits beside the trivial control, never alone.
summary = (R.groupby(["corpus","cond_label"])[["FRR","FAR","AUC"]].mean().round(4))
summary["balanced_acc"] = (1 - (R.groupby(["corpus","cond_label"])["FRR"].mean()
                                + R.groupby(["corpus","cond_label"])["FAR"].mean()) / 2).round(4)
display(summary)

print("\nMargin of each condition over the trivial control, per corpus (mean per-writer AUC):")
piv = R.pivot_table(index=["corpus","writer"], columns="condition", values="AUC")
for corpus, g in piv.groupby(level=0):
    line = f"  {corpus:8s} D(control) {g['D'].mean():.4f} | "
    line += " | ".join(f"{c} {g[c].mean():.4f} ({g[c].mean()-g['D'].mean():+.4f})"
                       for c in ["A","B","C"] if c in g)
    print(line)

## 7. Hypothesis tests

H1 compares condition A against condition B: does estimating the metric per writer reduce error
relative to one global metric? H2 compares condition A against condition D: does the proposed
representation outperform statistics that cannot read a signature? Both are paired across writers,
tested by Wilcoxon signed-rank, corrected by the Holm step-down procedure across the family, and
accompanied by bootstrap confidence intervals over writers so that the size of any difference is
visible alongside its significance.

In [ ]:
from scipy.stats import wilcoxon

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        run = max(run, val); adj[idx] = min(1.0, run)
    return adj

def boot_ci(d, n=N_BOOT, seed=RANDOM_STATE):
    r = np.random.RandomState(seed); d = np.asarray(d, float); d = d[~np.isnan(d)]
    if len(d) < 2: return (np.nan, np.nan)
    m = r.randint(0, len(d), (n, len(d)))
    bs = d[m].mean(1)
    return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

LOWER_IS_BETTER = {"FRR", "FAR"}
tests = []
for corpus, g in R.groupby("corpus"):
    piv = {c: g[g.condition == c].set_index("writer") for c in g.condition.unique()}
    for metric in ["AUC", "FRR", "FAR"]:
        for (c1, c2, hyp) in [("A","B","H1 per-writer vs global"), ("A","D","H2 proposed vs control")]:
            if c1 not in piv or c2 not in piv: continue
            common = piv[c1].index.intersection(piv[c2].index)
            x = piv[c1].loc[common, metric].values; y = piv[c2].loc[common, metric].values
            ok = ~(np.isnan(x) | np.isnan(y))
            x, y = x[ok], y[ok]
            diff = (y - x) if metric in LOWER_IS_BETTER else (x - y)   # positive = c1 better
            if len(x) < 5 or np.allclose(diff, 0):
                p = np.nan; stat = np.nan
            else:
                stat, p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
            lo, hi = boot_ci(diff)
            tests.append(dict(corpus=corpus, hypothesis=hyp, metric=metric,
                              n_writers=int(len(x)), mean_diff=float(np.mean(diff)),
                              ci_lo=lo, ci_hi=hi, stat=stat, p_raw=p))
T = pd.DataFrame(tests)
mask = T.p_raw.notna()
T.loc[mask, "p_holm"] = holm(T.loc[mask, "p_raw"].values)
T["significant"] = T.p_holm < 0.05
display(T.round(4))
print("\nPositive mean_diff means condition A is better on that metric.")

## 8. The metric, drawn

The figure below is the contribution made visible. Each writer's estimated weighting is shown
across the seventeen dissimilarity components. Two writers from the same corpus, enrolled by the
same procedure, produce visibly different weightings, because consistency differs by person and
the estimate reads it from the person's own signatures. No system in the reviewed literature can
produce this figure, because none holds a per-writer metric to draw.

In [ ]:
import matplotlib.pyplot as plt

WA = pd.DataFrame({ (R.loc[i,"corpus"], R.loc[i,"writer"]): W.loc[i]
                    for i in R.index[R.condition == "A"] }).T
WA.columns = PAIR_COMPONENTS

sd = WA.std(axis=0)
print("Between-writer variation in the PROFILE (sd of weight across writers):")
display(sd.round(3).to_frame("sd_across_writers").T)
print(f"mean sd across components: {sd.mean():.4f}   max: {sd.max():.4f}")

scales = R.loc[R.condition == "A", ["corpus","writer","scale"]]
print("\nBetween-writer variation in the SCALE (overall tightness):")
display(scales.groupby("corpus")["scale"].agg(["mean","std","min","max"]).round(4))

PROFILE_FLOOR = 0.02
if sd.max() < PROFILE_FLOOR:
    print(f"\n*** Profiles are near-identical across writers (max sd {sd.max():.4f} < {PROFILE_FLOOR}).")
    print("    Writers differ in overall tightness but not in which components vary.")
    print("    The per-writer weighting is then doing no work beyond what the kernel bandwidth")
    print("    already absorbs, and H1 should be expected to be retained. Reported, not hidden.")
else:
    # the two writers whose profiles differ most
    from itertools import combinations
    best, bd = None, -1.0
    idx = list(WA.index)
    for a, b in combinations(range(len(idx)), 2):
        d = float(np.abs(WA.iloc[a].values - WA.iloc[b].values).mean())
        if d > bd: bd, best = d, (idx[a], idx[b])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, key in zip(axes, best):
        v = WA.loc[key]
        ax.bar(range(len(v)), v.values, color="#1F3864")
        ax.axhline(1.0, color="#B00020", lw=1.0, ls="--")
        ax.set_xticks(range(len(v)))
        ax.set_xticklabels(PAIR_COMPONENTS, rotation=70, fontsize=7, ha="right")
        ax.set_title(f"{key[0]} / {key[1]}", fontsize=10)
        ax.spines[["top","right"]].set_visible(False)
    axes[0].set_ylabel("Estimated weight\n(1.0 = population average)")
    fig.suptitle("Two writers, two different similarity metrics, "
                 "estimated from their own signatures", fontsize=11)
    plt.tight_layout(); plt.show()
    print(f"\nmost dissimilar pair shown; mean absolute profile difference {bd:.4f}")

In [ ]:
R.drop(columns=["cond_label"]).to_csv("v9_per_writer_results.csv", index=False)
T.to_csv("v9_hypothesis_tests.csv", index=False)
WA.to_csv("v9_estimated_metrics.csv")
scales.to_csv("v9_metric_scales.csv", index=False)
print("written: v9_per_writer_results.csv, v9_hypothesis_tests.csv, v9_estimated_metrics.csv")

## 9. Confirmation run

The reserved writers have not been read by any notebook in this project. They are evaluated once,
at the end, after every design decision is final, and the result of that single run is what the
manuscript reports as the confirmation. Setting the flag in the configuration cell to True and
re-running this section consumes that opportunity; it cannot be undone, and it should not be run
while anything about the method is still being adjusted.

In [ ]:
if RUN_CONFIRMATION:
    conf = []
    for cond, rep in CONDITIONS:
        gc, gs = GLOBALS[rep]
        for k in RES_KEYS:
            conf.append(evaluate_writer(k, rep, cond, gc, gs))
    RC = pd.DataFrame(conf); RC.pop("weights")
    display(RC.groupby(["corpus","condition"])[["FRR","FAR","AUC"]].mean().round(4))
    RC.to_csv("v9_confirmation_results.csv", index=False)
    print("confirmation written. This run is final.")
else:
    print("RUN_CONFIRMATION is False. Reserved writers untouched:", len(RES_KEYS))

## 10. Reading the result

Condition A is the proposal; B isolates the per-writer decision; C shows what the representation
achieves with no weighting at all; D is the floor that any claim about reading signatures must
clear.

If A beats D by a margin whose confidence interval excludes zero, the representation is reading
the signature rather than the artifact, and H2 is rejected in the study's favour. If A does not
beat D, that is reported as the result, because an accuracy above a control that cannot read
signatures is the only accuracy that means anything on these corpora.

If A beats B, estimating the metric per writer is doing work that one global metric cannot, and
H1 is rejected in the study's favour. If it does not, the honest finding is that the writer-specific
component of this problem does not lie in the metric either, and the contribution rests on the
system and the audit rather than on the estimator. Both outcomes are reportable; only one is
discoverable in advance, and it is not this notebook's job to prefer either.